# Part 3 - Advanced Feature Engineering
Run Part 1 first so `train` and `test` are loaded.

In [ ]:
import numpy as np
import pandas as pd

TARGET='addicted_label'
ID='id'

train_fe=train.copy()
test_fe=test.copy()


## Helper Function

In [ ]:
def create_features(df):

    eps=1e-3

    # Total digital activity
    df["total_usage"] = (
        df["social_media_hours"] +
        df["gaming_hours"] +
        df["work_study_hours"]
    )

    # Ratios
    df["screen_sleep_ratio"] = df["daily_screen_time_hours"]/(df["sleep_hours"]+eps)
    df["social_ratio"] = df["social_media_hours"]/(df["daily_screen_time_hours"]+eps)
    df["gaming_ratio"] = df["gaming_hours"]/(df["daily_screen_time_hours"]+eps)
    df["work_ratio"] = df["work_study_hours"]/(df["daily_screen_time_hours"]+eps)

    df["notifications_per_hour"] = df["notifications_per_day"]/(df["daily_screen_time_hours"]+eps)
    df["opens_per_hour"] = df["app_opens_per_day"]/(df["daily_screen_time_hours"]+eps)

    df["weekend_extra"] = df["weekend_screen_time"]-df["daily_screen_time_hours"]

    df["usage_per_age"] = df["daily_screen_time_hours"]/(df["age"]+eps)

    df["stress_usage"] = (
        df["daily_screen_time_hours"] *
        df["stress_level"].map({
            "Low":1,
            "Medium":2,
            "High":3
        }).fillna(2)
    )

    df["sleep_deficit"]=(8-df["sleep_hours"]).clip(lower=0)

    df["high_screen_flag"]=(df["daily_screen_time_hours"]>8).astype(int)
    df["poor_sleep_flag"]=(df["sleep_hours"]<6).astype(int)
    df["high_notification_flag"]=(df["notifications_per_day"]>150).astype(int)

    df["age_bucket"]=pd.cut(
        df["age"],
        bins=[0,18,25,35,50,100],
        labels=["Teen","18-25","26-35","36-50","50+"]
    )

    return df


In [ ]:
train_fe=create_features(train_fe)
test_fe=create_features(test_fe)

print("Original Features :", train.shape[1]-2)
print("Engineered Features :", train_fe.shape[1]-2)
display(train_fe.head())


## Missing Value Handling

In [ ]:
FEATURES=[c for c in train_fe.columns if c not in [ID,TARGET]]

CAT_FEATURES=train_fe[FEATURES].select_dtypes(
    include=["object","category"]
).columns.tolist()

NUM_FEATURES=[c for c in FEATURES if c not in CAT_FEATURES]

for c in CAT_FEATURES:
    train_fe[c]=train_fe[c].fillna("Missing")
    test_fe[c]=test_fe[c].fillna("Missing")

for c in NUM_FEATURES:
    med=train_fe[c].median()
    train_fe[c]=train_fe[c].fillna(med)
    test_fe[c]=test_fe[c].fillna(med)

print("Categorical:",len(CAT_FEATURES))
print("Numerical :",len(NUM_FEATURES))


## Quick CV Check

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from catboost import CatBoostClassifier
import numpy as np

X=train_fe.drop(columns=[ID,TARGET])
y=train_fe[TARGET]
X_test=test_fe.drop(columns=[ID])

cat_idx=[X.columns.get_loc(c) for c in CAT_FEATURES]

cv=StratifiedKFold(n_splits=5,shuffle=True,random_state=42)

scores=[]

for fold,(tr,va) in enumerate(cv.split(X,y),1):

    model=CatBoostClassifier(
        iterations=500,
        learning_rate=0.05,
        depth=8,
        loss_function="Logloss",
        eval_metric="AUC",
        verbose=False,
        random_seed=42
    )

    model.fit(
        X.iloc[tr],
        y.iloc[tr],
        cat_features=cat_idx,
        eval_set=(X.iloc[va],y.iloc[va]),
        use_best_model=True
    )

    pred=model.predict_proba(X.iloc[va])[:,1]
    auc=roc_auc_score(y.iloc[va],pred)
    scores.append(auc)

    print(f"Fold {fold}: {auc:.6f}")

print("\nMean CV:",np.mean(scores))


## Next
In Part 4 we'll build a fully optimized CatBoost pipeline with:
- Out-of-fold predictions
- Test prediction averaging
- Feature importance
- Submission generation
- Tunable parameters
